In [1]:
%pip install pandas
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd 
import numpy as np
import re

In [3]:
data = pd.read_excel(r'C:\Users\KS\Desktop\K4รายงานสต็อกการ์ด พร้อมทุน.xls' ,engine='calamine',header=11,usecols="A:F",dtype={'Unnamed: 3': str})
data.rename(columns={'Unnamed: 0':'DATE','Unnamed: 1':'Bill','เพิ่ม ':'details','Unnamed: 3':'value','ลด ':'export','คงเหลือ ':'balance'} ,inplace=True)

In [4]:
data['product_id'] = data.loc[data['DATE'] =='รหัสสินค้า', 'export']
data['product_id'] = data['product_id'].ffill()

In [5]:
data['unit'] = data.loc[data['DATE'].astype(str).str.strip() == 'คลัง', 'balance']
data['unit'] = data['unit'].ffill()
data['unit'] = data['unit'].str.extract(r'(\d+)').fillna(0).astype(int)

In [6]:
# แปลงตรงๆ โดยบอกสไตล์ปฏิทินสากลไปก่อน
data['DATE'] = pd.to_datetime(data['DATE'], format='%d/%m/%Y', errors='coerce')

# ลบปีออก 543 ปี (ใช้ DateOffset)
data['DATE'] = data['DATE'] - pd.DateOffset(years=543)
data['DATE'] = data['DATE'].dt.date

In [7]:
data.dropna(subset=['DATE'], inplace=True)

In [8]:
# 1. บังคับชุบชีวิตข้อมูลในคอลัมน์ให้กลายเป็นข้อความ (String) ชัวร์ๆ 100% ก่อน
data['value_str'] = data['value'].astype(str)

# 2. ก่อนรัน Regex เติมจุดทศนิยม .0 เข้าไปเฉพาะตัวที่เป็นจำนวนเต็ม (เช่น '5' -> '5.0')
# เพื่อป้องกันไม่ให้ Regex ตัวหน้าและตัวหลังพัง
data['value_str'] = data['value_str'].apply(lambda x: x if '.' in x else x + '.0')

# 3. เอาตัวเลขหน้าจุด (ใช้ตัวแปร value_str ที่เป็นข้อความแล้ว)
data['front_value'] = data['value_str'].str.extract(r"^([0-9]+)\.").fillna(0).astype(int)

# 4. เอาตัวเลขหลังจุด (ข้าม 0 หน้า เจอเลข 1-9 ปุ๊บ กวาดขวาหมดเท่าที่มีจริง)
data['back_value'] = data['value_str'].str.extract(r"\.0*([1-9].*)").fillna(0).astype(int)

In [9]:
data['import'] = data['back_value'] + (data['unit'] * data['front_value']) 

 Robust Z-score

In [10]:
#%pip install scipy
#%pip install python-calamine
#%pip install pandas
#%pip install openpyxl
#%pip install rapidfuzz

In [11]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import openpyxl

# ==========================================
# 1. โหลดข้อมูล (ใส่ engine='calamine' เพื่อเลี่ยง XML เสียหายจากรอบแรก)
# ==========================================

df = data[['DATE', 'Bill', 'details', 'product_id', 'import']].copy()

# ล้างช่องว่างที่อาจมองไม่เห็นในชื่อคอลัมน์ทิ้งให้หมดเพื่อความปลอดภัย
df.columns = df.columns.str.strip()

# [เสริมเกราะ 1] แปลง ID ให้เป็น string ทั้งหมด ป้องกันกรณี Excel แปลงบางตัวเป็นตัวเลขแล้วกลุ่มเพี้ยน
df['product_id'] = df['product_id'].astype(str).str.strip()

# เอาเฉพาะยอดนำเข้าที่มากกว่า 0 เท่านั้น (ตัด Noise/บิลยกเลิก ออก)
df = df[df['import'] > 0]

# ==========================================
# 2. คำนวณหา Outlier 
# ==========================================
# หา IQR และคูณสเกล 1.4826 สำหรับ แผน A
q1 = df.groupby('product_id')['import'].transform(lambda x: x.quantile(0.25))
q3 = df.groupby('product_id')['import'].transform(lambda x: x.quantile(0.75))
iqr_scaled = (q3 - q1) * 1.4826

# หา Median และจำนวนบิล
group_median = df.groupby("product_id")["import"].transform("median")
group_count = df.groupby('product_id')['import'].transform('count')
group_sd = df.groupby("product_id")["import"].transform("std")
# ✨ [แก้ไขจุดที่ 1] หา MAD ดิบจาก SciPy แล้วค่อยคูณสเกล 1.4826 ข้างนอก 
mad_raw = df.groupby("product_id")["import"].transform(stats.median_abs_deviation)
group_mad_scaled = mad_raw * 1.4826

# เงื่อนไขที่ 2 ไม่ให้สูงเกิน 30% ของค่ากลาง
max_allowed_deviation = np.maximum(group_median * 0.3, 1.0)
group_mad_scaled = np.minimum(group_mad_scaled, max_allowed_deviation)
group_mad_scaled = np.maximum(group_mad_scaled, 1.0) # กันตัวหารเป็น 0

# เงื่อนไขการแบ่งกลุ่มสินค้า
conditions = [
    (iqr_scaled > 0) & (group_count >= 10),   # แผน A
    (iqr_scaled == 0) | (group_count < 10)    # แผน B
]

# คำนวณคะแนนดิบ Z-score
df["Adaptive_ZScore"] = np.select(conditions, [
    ((df["import"] - group_median) / iqr_scaled),       # แผน A 
    ((df["import"] - group_median) / group_mad_scaled)  # แผน B 
], default=0)

# min max mean 
#df['min'] = df.groupby('product_id')['import'].transform('min')
#df['max'] = df.groupby('product_id')['import'].transform('max')
#df['mean'] = df.groupby('product_id')['import'].transform('mean')
#df["z_score"] = (df["import"] - df['mean']) / group_sd
#df["median"] = group_median
#df["IQR"] = (q3 - q1)


# ==============================================================================
# Empirical Rule / Standard Deviation (SD) Coverage (Normal Distribution)
# ==============================================================================

# ± 0.5 SD = ครอบคลุมข้อมูล 38.29% (โอกาสหลุดเกณฑ์ ~ 61.71%)
# ± 1.0 SD = ครอบคลุมข้อมูล 68.27% (โอกาสหลุดเกณฑ์ ~ 31.73%)
# ± 1.5 SD = ครอบคลุมข้อมูล 86.64% (โอกาสหลุดเกณฑ์ ~ 13.36%)
# ± 2.0 SD = ครอบคลุมข้อมูล 95.45% (โอกาสหลุดเกณฑ์ ~  4.55%)
# ± 2.5 SD = ครอบคลุมข้อมูล 98.76% (โอกาสหลุดเกณฑ์ ~  1.24%)
# ± 3.0 SD = ครอบคลุมข้อมูล 99.73% (โอกาสหลุดเกณฑ์ ~  0.27% -> Extreme / Outlier)
# ± 3.5 SD = ครอบคลุมข้อมูล 99.95% (โอกาสหลุดเกณฑ์ ~  0.05%)
# ± 4.0 SD = ครอบคลุมข้อมูล 99.99% (โอกาสหลุดเกณฑ์ ~  0.01%)
df["is_outlier"] = df['Adaptive_ZScore'].abs() > 1.5

# ==========================================
# 3. เจาะลึกระดับบิล (Drill-Down)
# ==========================================
# ดึงรายชื่อรหัสสินค้าทั้งหมดที่มีแถวใดแถวหนึ่งติดสถานะ Outlier
final_report = df[df["is_outlier"] == True].copy()

# ✨ [แก้ไขจุดที่ 2] จัดเรียงข้อมูลพร้อมใส่วงเล็บปิดให้สมบูรณ์
final_report = final_report.sort_values(
    by=["product_id", "Adaptive_ZScore"], ascending=[True, False]
)

In [12]:
final_report.to_excel(r'C:\Users\KS\Desktop\ชื่อ K4ตั้งแต่มีนา.xlsx',index=False,engine='openpyxl')

In [13]:
search_terms = pd.read_excel(r'C:\Users\KS\Desktop\ชื่อเค4.xlsx' ,engine='calamine',usecols=['ชื่อสินค้า ผิดจากระบบ'])
search_terms = search_terms['ชื่อสินค้า ผิดจากระบบ'].dropna().tolist()

In [14]:
from rapidfuzz import process, fuzz

# ฟังก์ชัน Clean ข้อความให้สม่ำเสมอ
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()  # ตัวพิมพ์เล็ก
    text = re.sub(r'[^\w\s]', '', text)  # ลบเครื่องหมายพิเศษ เช่น . () -
    return text.strip()

# Clean ข้อมูลก่อนนำไปประมวลผล
cleaned_targets = [clean_text(t) for t in search_terms]

def get_max_similarity_fast(product_name, targets):
    cleaned_name = clean_text(product_name)
    if not cleaned_name or not targets:
        return 0
    
    # process.extractOne จะหาคำที่คล้ายที่สุดให้อัตโนมัติแบบ C-Speed
    # ใช้ scorer=fuzz.WRatio เพื่อความแม่นยำระดับโปร
    match = process.extractOne(cleaned_name, targets, scorer=fuzz.WRatio)
    
    return match[1] if match else 0  # match[1] คือคะแนนความคล้าย

# 4. คำนวณคะแนนความคล้าย
final_report['max_similarity'] = final_report['product_id'].apply(
    lambda x: get_max_similarity_fast(x, cleaned_targets)
)

# 5. กรองเฉพาะรายการที่คะแนนถึงเกณฑ์
threshold = 70
filtered_df = final_report[final_report['max_similarity'] >= threshold].copy()


In [15]:
filtered_df[['DATE', 'Bill', 'details', 'product_id', 'import', 'Adaptive_ZScore','max_similarity']].to_excel(
    r'C:\Users\KS\Desktop\product_out1.xlsx',index=False,engine='openpyxl')

In [16]:
#final_report[['DATE','Bill','product_id','import','min','max','mean','median','IQR','z_score','Adaptive_ZScore']].reset_index(drop=True).to_excel(
    #r'C:\Users\KS\Desktop\product_out1.xlsx',index=False,engine='openpyxl')
                                                                                    